# 03 · Frozen test set + training simulations (CPU session)

Resumable: if the session dies, attach this notebook's previous output, copy it back into /kaggle/working and rerun (finished shards are skipped).

Output: `shards/test/<noise>/` and `shards/train/` (~4.5 GB for 2M stars).

In [ ]:
# --- Setup: get the code (no project logic in notebooks; everything lives in src/ and scripts/) ---
import os
import subprocess
import sys

REPO_URL = "https://github.com/powershell1/Almanac.git"
BRANCH = "master"
SRC = "/tmp/Almanac"
if os.path.exists("/kaggle/input/almanac-src/pyproject.toml"):
    SRC = "/kaggle/input/almanac-src"          # offline option: repo uploaded as a Dataset
elif not os.path.exists(SRC):
    url = REPO_URL
    try:                                       # private repo: add a Kaggle Secret GITHUB_TOKEN
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, SRC], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zuko==1.6.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
WORK = "/kaggle/working"
CFG = "configs/killtest.yaml"


In [ ]:
CACHE = "/kaggle/input/gdr4-02-scan-cache/data_cache/scan_cache_dr4_gaiamock.npz"
# Optional resume: copy shards from a previous version of this notebook
PREV = "/kaggle/input/gdr4-03-simulate/shards"
if os.path.exists(PREV):
    subprocess.run(["cp", "-rn", PREV, WORK], check=True)

In [ ]:
!python scripts/make_test_set.py --config {CFG} --root {WORK} --set paths.scan_cache={CACHE}

In [ ]:
!python scripts/simulate.py --config {CFG} --root {WORK} --set paths.scan_cache={CACHE}

In [ ]:
!du -sh /kaggle/working/shards/*